# KeyLLM Testing
Test the KeyLLM

In [1]:
import os
from dotenv import load_dotenv

import openai
from keybert.llm import OpenAI
from keybert import KeyLLM

load_dotenv("../.env")          

doc = """
Relative Contribution Of Trees And Crops To Soil Carbon Content In A Parkland System In Burkina Faso Using Variations In Natural C-13 Abundance","The Origin Of Organic Matter Was Studied In The Soils Of A Parkland Of Karite (Vitallaria Paradoxa C.F. Gaertn) And Nere (Parkia Biglobosa (Jacq.) Benth.), Which Is Extensively Cultivated Without The Use Of Fertilisers. In Such Systems, Fertility (Physical, Chemical And Biological) Gradients Around Trees Have Been Attributed By Some Authors To A Priori Differences In Fertility, Allowing For Better Tree Establishment On Richer Sites. In Reverse, Other Workers Believed That These Gradients Are Due To The Contribution Of Trees To The Formation Of Soil Organic Matter Through Litter And Decay Of Roots. Measurements Of The Variations In The C-13 Isotopic Composition Allowed For A Distinction Between Tree (C-3) Derived C And Crop And Grass (C-4) Derived C In The Total Soil Organic C Content. The Organic Carbon Contents Of The Soils Were Recorded Under The Two Species At Two Soil Depths And At Five Distances Going From Tree Trunk To The Open Area And Their C Isotopic Signatures Were Analysed. The Results Showed That Soil Carbon Contents Under Karite (6.43 +/- 0.45 G Kg(-1)) And Nere (5.65 +/- 0.27 G Kg(-1)) Were Significantly Higher (P < 0.01) Than In The Open Area (4.09 +/- 0.26 G Kg(-1)). The Delta C-13 Of Soil C Was Significantly Higher (P < 0.001) In The Open Area (-17.5 +/- 0.3 Parts Per Thousand) Compared With The Values Obtained On Average With Depth And Distance From Tree Under Karite (-20.2 +/- 0.4 Parts Per Thousand) And Nere (-20.1 +/- 0.4 Parts Per Thousand). The C-4-Derived Soil C Was Approximately Constant, And The Differences In Total Soil C Were Fully Explained By The C-3 (Tree) Contributions To Soil Carbon Of 4.01 +/- 0.71, 3.02 +/- 0.53, 1.53 +/- 0.10 G Kg(-1), Respectively Under Karite, Nere And In The Open Area. These Results Show That Trees In Parklands Have A Directly Positive Contribution To Soil Carbon Content, Justifying The Need To Encourage The Maintenance Of Trees In These Systems In Semi-Arid Environments Where The Carbon Content Of Soil Appears To Be The First Limiting Factor For Crop Growth.
"""

client = openai.OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

llm = OpenAI(
    client,
    model="openai/gpt-5.6-luna",
    chat=True,
    delay_in_seconds=3,
    exponential_backoff=True,
    verbose=True,
)

# Load it in KeyLLM
kw_model = KeyLLM(llm)
# Extract keywords
keywords = kw_model.extract_keywords(doc)

print(keywords)

/home/li422/repos/isric/metadata-augmentation/keyword-extractor/.venvipynb/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
0it [00:03, ?it/s]


APIStatusError: Error code: 402 - {'error': {'message': "This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 29873. To increase, visit https://openrouter.ai/workspaces/default/keys/f2402f069090db287830073eb0ffc7a298604b7507d02d3b82bc5a708c3dad14 and adjust the key's total limit", 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.', 'provider_name': None, 'previous_errors': [{'code': 402, 'message': "This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 32860. To increase, visit https://openrouter.ai/workspaces/default/keys/f2402f069090db287830073eb0ffc7a298604b7507d02d3b82bc5a708c3dad14 and adjust the key's total limit"}, {'code': 402, 'message': "This request requires more credits, or fewer max_tokens. You requested up to 65

In [7]:
# Create your LLM
prompt = """
I have the following document:
[DOCUMENT]

Based on the information above, extract the keywords that best describe the topic of the text and related with soil.
Separate the keywords with comma.
"""

llm = OpenAI(
    client,
    model="openai/gpt-5.6-luna",
    chat=True,
    delay_in_seconds=3,
    exponential_backoff=True,
    verbose=True,
    prompt = prompt
)

# Load it in KeyLLM
kw_model = KeyLLM(llm)

# Extract keywords
keywords = kw_model.extract_keywords(doc); keywords

1it [00:06,  6.73s/it]


[['Soil carbon',
  'soil organic carbon',
  'soil organic matter',
  'soil fertility',
  'carbon sequestration',
  'carbon-13 isotopic composition',
  'C3-derived carbon',
  'C4-derived carbon',
  'tree-derived carbon',
  'soil carbon content',
  'soil fertility gradients',
  'litter decomposition',
  'root decay',
  'parkland agroforestry',
  'semi-arid soils',
  'Burkina Faso',
  'karite',
  'nere']]

In [12]:
# Fine-tune Candidate Keywords

import json

with open("../concepts.json", encoding="utf-8") as f:
    concepts = json.load(f)
soilvoc = [label for concept in concepts for label in concept["labels"]
.get("en", [])] # get all en labels
soilvoc = list(set(soilvoc)) # get unique values
soilvoc = [s.lower() for s in soilvoc]

# Create your LLM
prompt = """
I have the following document:
[DOCUMENT]

With the following candidate keywords:
[CANDIDATES]

Based on the information above, improve the candidate keywords to best describe the topic of the document.

Separate the keywords with comma.
"""

llm = OpenAI(
    client,
    model="openai/gpt-5.6-luna",
    chat=True,
    delay_in_seconds=3,
    exponential_backoff=True,
    verbose=True,
    prompt = prompt
)

# Load it in KeyLLM
kw_model = KeyLLM(llm)


keywords = kw_model.extract_keywords(doc, candidate_keywords=[soilvoc]); keywords

1it [00:08,  8.38s/it]


[['soil organic carbon',
  'soil total carbon',
  'soil organic matter',
  'soil organic matter content',
  'soil carbon density',
  'carbon pools',
  'carbon storage capacity',
  'trees',
  'tree',
  'crop',
  'grasses',
  'vegetation types',
  'vegetation',
  'land use',
  'soil fertility',
  'soil nutrient content',
  'soil productivity',
  'root biomass',
  'soil depth sampled',
  'soil spatial property']]

In [13]:
doc_de = """
Die Gesamt-Phosphoreinträge in die Gewässer wurden mit dem Stoffflussmodell MODIFFUS über alle diffusen Eintragsquellen (Ackerland, Dauergrünland, Wald, Gletscher, Siedlungsgrünflächen etc.) und alle diffusen Eintragspfade (Bodenerosion, Auswaschung, Abschwemmung, Drainage, atmosphärische Deposition etc.) berechnet. Die Karte zeigt die aufsummierten Verluste pro Landnutzungskategorie im Hektarraster, basierend auf der Arealstatistik 2013/18. Es wurden mittlere klimatische Bedingungen zugrunde gelegt, das Bezugsjahr ist 2020.</p><p> 
"""
keywords = kw_model.extract_keywords(doc_de, candidate_keywords=[soilvoc]); keywords

1it [00:10, 10.13s/it]


[['phosphorus',
  'phosphorus total elements',
  'soil phosphorus loss',
  'soil p loss',
  'soil nutrient loss',
  'land use class',
  'land cover',
  'land use',
  'arable land development',
  'land use forest',
  'land use grasses',
  'land use shrubs',
  'vegetation types',
  'surface runoff',
  'soil erosion',
  'soil erosion area affected',
  'soil erosion degree',
  'soil leaching',
  'drainage to groundwater',
  'interflow',
  'precipitation',
  'weather conditions',
  'climate',
  'climatic effects',
  'soil water flow',
  'water quality',
  'nutrient dynamics',
  'MODIFFUS',
  'diffuse pollution',
  'hectare grid',
  'areal statistics',
  '2020 reference year']]